In [11]:
import pandas as pd
import numpy as np
import torch
import random
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sdv.single_table import TVAESynthesizer as TVAE
from sdv.metadata import SingleTableMetadata

In [12]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

In [13]:
dir_datasets = '/kaggle/input/nir-data/data/processed/'

# Загрузка реальных датасетов
real_data_1 = pd.read_csv(dir_datasets+'abolone.csv')
real_data_2 = pd.read_csv(dir_datasets+'insurance.csv')
real_data_3 = pd.read_csv(dir_datasets+'two_moons.csv')

# Словарь датасетов для удобства 
datasets = {
    'abolone': real_data_1,
    'insurance': real_data_2,
    #'two_moons': real_data_3
}

for name, data in datasets.items():
    print(f"{name}: {data.shape}, колонки: {list(data.columns)}")

abolone: (4177, 9), колонки: ['Sex', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight', 'Rings']
insurance: (1338, 7), колонки: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'expenses']


In [14]:
# Определение пространства поиска гиперпараметров для TVAE
tvae_space = {
    'batch_size': hp.choice('batch_size', [100]),
    'embedding_dim': hp.choice('embedding_dim', [16, 32, 64]),
    'compress_dims': hp.choice('compress_dims', [[256, 256], [512, 512]]),
    'decompress_dims': hp.choice('decompress_dims', [
        [256, 256], [512, 512],
        [256, 256, 256, 256],
        [512, 512, 512, 512]
    ]),
    'loss_factor': hp.choice('loss_factor', [2, 3]),
    'l2scale': hp.qloguniform('l2scale', np.log(1e-5), np.log(6.3e-5), 1e-6),
    'epochs': hp.choice('epochs', [400])
}

In [15]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

def evaluate_c2st(real, synthetic):
    real_copy = real.copy()
    synthetic_copy = synthetic.copy()
    
    # Добавление меток
    real_copy['label'] = 1
    synthetic_copy['label'] = 0
    
    # Объединение данных
    df = pd.concat([real_copy, synthetic_copy], ignore_index=True)
    
    # Определение категориальных колонок
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
    categorical_columns = [col for col in categorical_columns if col != 'label']
    
    # Разделение на признаки и метки
    X = df.drop(columns='label')
    y = df['label']
    
    # Разделение на обучающую и тестовую выборки
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=RANDOM_SEED
    )
    
    # Обучение CatBoost
    clf = CatBoostClassifier(
        random_seed=RANDOM_SEED,
        verbose=0
    )
    
    clf.fit(X_train, y_train, cat_features=categorical_columns, eval_set=(X_test, y_test))
    
    # Предсказание и вычисление метрик
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    return f1 #accuracy


In [16]:
def generate_and_evaluate(params, dataset):
    #Извлечение метаданных
    metadata = SingleTableMetadata()
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    metadata.detect_from_dataframe(data=dataset)
    
    
    # Создание и обучение TVAE с заданными параметрами
    tvae = TVAE(
        metadata=metadata,
        batch_size=params['batch_size'],
        embedding_dim=params['embedding_dim'],
        compress_dims=params['compress_dims'],
        decompress_dims=params['decompress_dims'],
        l2scale=params['l2scale'],
        loss_factor=params['loss_factor'],
        #epochs=int(params['epochs'])
    )
    
    # Обучение модели на реальных данных
    tvae.fit(dataset)
    
    # Генерация синтетических данных
    synthetic_data = tvae.sample(len(dataset))
    
    # Вычисление метрики C2ST
    c2st_score = evaluate_c2st(dataset, synthetic_data)

    return c2st_score

In [17]:
def optimize_dataset(dataset_name, dataset, max_evals=30):
    print(f"Оптимизация для датасета: {dataset_name}, размер: {dataset.shape}")
    
    # Создаем объект для хранения истории поиска
    trials = Trials()
    
    # Определяем целевую функцию для оптимизации
    def objective(params):
        print("="*50)
        c2st_score = generate_and_evaluate(params, dataset)

        print(c2st_score)
        print(params)
        print("="*50)
        # Возвращаем словарь в формате, который ожидает hyperopt
        return {
            'loss': c2st_score,  
            'status': STATUS_OK,
            'c2st_score': c2st_score  
        }
    
    
    # Запускаем оптимизацию с помощью TPE алгоритма
    rng = np.random.default_rng(RANDOM_SEED)
    best = fmin(
        fn=objective,
        space=tvae_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=rng
    )
    
    # Получаем лучшие параметры в читаемом виде
    best_params = {
        'batch_size': 100,
        'embedding_dim': [16, 32, 64][best['embedding_dim']],
        'compress_dims': [[256, 256], [512, 512]][best['compress_dims']],
        'decompress_dims': [
            [256, 256], [512, 512],
            [256, 256, 256, 256],
            [512, 512, 512, 512]
        ][best['decompress_dims']],
        'loss_factor': [2, 3][best['loss_factor']],
        'l2scale': best['l2scale'],
        'epochs': float('inf')
    }
    
    # Находим лучший результат
    best_trial = min(trials.trials, key=lambda x: x['result']['loss'])
    best_c2st = best_trial['result'].get('c2st_score', None)
    
    results = {
        'best_params': best_params,
        'best_c2st': best_c2st,
        'c2st_diff_from_optimal': best_trial['result']['loss']
    }
    
    print(f"Лучший C2ST для {dataset_name}: {best_c2st}")
    print(f"Лучшие параметры: {best_params}")
    print('-' * 50)
    
    return results

In [18]:
# Словарь для хранения результатов
all_results = {}

# Указываем количество итераций для каждого датасета
max_evals = 50 

# Запускаем оптимизацию для каждого датасета
for dataset_name in datasets:
    data = datasets[dataset_name]
    all_results[dataset_name] = optimize_dataset(dataset_name, data, max_evals)

Оптимизация для датасета: abolone, размер: (4177, 9)
0.9908838684106223                                    
{'batch_size': 100, 'compress_dims': (512, 512), 'decompress_dims': (512, 512), 'embedding_dim': 64, 'epochs': 400, 'l2scale': 3.1e-05, 'loss_factor': 2}
0.9877324891175306                                                                 
{'batch_size': 100, 'compress_dims': (512, 512), 'decompress_dims': (512, 512, 512, 512), 'embedding_dim': 16, 'epochs': 400, 'l2scale': 3.1e-05, 'loss_factor': 3}
0.988950276243094                                                                  
{'batch_size': 100, 'compress_dims': (512, 512), 'decompress_dims': (256, 256), 'embedding_dim': 16, 'epochs': 400, 'l2scale': 1.1e-05, 'loss_factor': 2}
0.9932512901945216                                                                 
{'batch_size': 100, 'compress_dims': (512, 512), 'decompress_dims': (256, 256, 256, 256), 'embedding_dim': 32, 'epochs': 400, 'l2scale': 1.4999999999999999e-05, 'loss_f